## 11.4 GRU - Pytorch 实现

#### 1. 这一节的核心目标

##### 1.1 这一节要真正理解的 4 件事

- PyTorch 如何高效实现 GRU 的三个门
- 为什么权重矩阵的第一维是 $3 \times hidden\_size$
- `output / h_n` 的本质分别是什么
- 多层与双向 GRU 的 shape 如何变化

#### 2. PyTorch 中最基础的 GRU 用法

##### 2.1 最基本的写法

这里：

- `input_size=4`：每个时间步输入特征维度是 4
- `hidden_size=5`：隐藏状态维度是 5
- `batch_first=True`：输入输出格式按 `(B, T, D)` 来组织

In [ ]:
import torch
import torch.nn as nn

gru = nn.GRU(
    input_size=32, # 输入特征维度
    hidden_size=64, # 隐藏状态维度
    batch_first=True, # 输入输出的batch维度在第一维
)

X = torch.randn(100, 10, 32) # (batch_size, seq_len, input_size)

output, h_n = gru(X) # output: (batch_size, seq_len, hidden_size) h_n: (num_layers, batch_size, hidden_size)

##### 2.2 输入张量 shape

表示：

- batch size = 2
- 序列长度 = 3
- 每个时间步特征维度 = 4

In [ ]:
X.shape # (100, 10, 32) 输入的形状

##### 2.3 输出张量 shape

In [ ]:
output.shape # (100, 10, 64) 最后一层中每个时间步的输出的汇总
h_n.shape # (num_layers, 100, 64) 每一层最后一个时间步的隐藏状态的汇总

#### 3. PyTorch 中 GRU 的核心优化思想

##### 3.1 理论上，GRU 有三组计算

我们前面从理论角度学过，GRU 单时间步要算：

- 更新门 $z_t$
- 重置门 $r_t$
- 候选隐藏状态 $\tilde{h}_t$

如果完全按照公式直译实现，看起来就像要分别做 3 次线性变换。

##### 3.2 但 PyTorch 不会这样“分开慢慢算”

在工程实现中，PyTorch 不会真的把三个门完全拆开逐个做独立矩阵乘法，而是会尽可能：

把多个线性变换合并成一次更大的矩阵运算，再切分。

这样做的好处是：

- 减少矩阵乘法次数
- 更适合 GPU 并行
- 提高计算效率
- 降低内存访问开销

这和我们前面学 LSTM 时的合并 4 次计算思想非常相似。

#### 4. 为什么 GRU 的权重是 3 倍大小

##### 4.1 先看 PyTorch 中的权重名字

```python
for name, param in gru.named_parameters():
    print(name, param.shape)
```

通常会看到：

```python
weight_ih_l0
weight_hh_l0
bias_ih_l0
bias_hh_l0
```

其中：

- `ih = input to hidden`
- `hh = hidden to hidden`
- `l0 = 第 0 层（第一层）`

##### 4.2 它们的 shape 是什么

如果：

- `input_size = 4`
- `hidden_size = 5`

那么通常会看到：

```python
weight_ih_l0.shape = (15, 4)
weight_hh_l0.shape = (15, 5)
bias_ih_l0.shape = (15,)
bias_hh_l0.shape = (15,)
```

因为：

$3 \times 5 = 15$

##### 4.3 为什么是 $3 \times hidden\_size$

这是因为 GRU 有三组门相关计算：

- 更新门
- 重置门
- 候选隐藏状态

每一组都需要输出一个长度为 `hidden_size` 的向量。

所以 PyTorch 会把这三组计算打包在一起：

$3d_h$

于是就得到：

- 输入权重：$(3d_h,\ d_x)$
- 隐藏权重：$(3d_h,\ d_h)$

##### 4.4 本质理解

GRU 的权重第一维是 3 倍，不是因为隐藏维度变大了，而是因为：

三组门控 / 候选状态计算，被合并进同一个大矩阵中。

#### 5. PyTorch 中单时间步的实现思路

##### 5.1 理论视角

从理论上看，单时间步需要分别算：

- $z_t$
- $r_t$
- $\tilde{h}_t$

但工程上会先把前两部分甚至更多部分合并起来计算。

##### 5.2 一个便于理解的伪代码

```python
# 输入
x_t      # (B, d_x)
h_prev   # (B, d_h)

# 一次性线性变换
gates_x = x_t @ W_ih.T + b_ih        # (B, 3*d_h)
gates_h = h_prev @ W_hh.T + b_hh     # (B, 3*d_h)

# 切分成三部分
x_z, x_r, x_n = gates_x.chunk(3, dim=1)
h_z, h_r, h_n = gates_h.chunk(3, dim=1)

# 更新门、重置门
z_t = sigmoid(x_z + h_z)
r_t = sigmoid(x_r + h_r)

# 候选隐藏状态
n_t = tanh(x_n + r_t * h_n)

# 最终隐藏状态
h_t = (1 - z_t) * n_t + z_t * h_prev
```

这里的 `n_t` 本质上就对应理论里的候选隐藏状态 $\tilde{h}_t$。

#### 6. output 和 h_n 到底是什么

##### 6.1 output 的本质

如果：

```python
output, h_n = gru(X)
```

那么：

```python
output.shape = (B, T, d_h)
```

这里的 `output` 表示：

最后一层 GRU 在每个时间步输出的隐藏状态序列。

也就是：

$[h_1,\ h_2,\ h_3,\ ...,\ h_T]$

如果 `batch_first=True`，则组织成：

$(B,\ T,\ d_h)$

##### 6.2 h_n 的本质

`h_n` 表示：

最后一个时间步的隐藏状态。

对于单层、单向 GRU：

```python
h_n.shape = (1, B, d_h)
```

第一维的 `1` 表示：

- 1 层
- 1 个方向

所以它本质上对应：

$h_T$

只是 PyTorch 会统一保留“层数 × 方向数”这一维。

##### 6.3 两者的区别

这个区别一定要牢牢记住：

- `output`：所有时间步的隐藏状态
- `h_n`：最后时间步的隐藏状态

可以理解成：

```python
output = [h1, h2, h3, ..., hT]
h_n    = hT
```

当然在多层、双向时会更复杂一些，但本质不变。

#### 7. 多层 GRU 的维度变化

##### 7.1 最基础定义

```python
gru = nn.GRU(
    input_size=4,
    hidden_size=5,
    num_layers=3,
    batch_first=True
)
```

输入仍然是：

```python
X.shape = (2, 3, 4)
```

##### 7.2 多层 GRU 的核心理解

你可以把多层 GRU 理解为：

时间维上递归，层维上堆叠。

在某个时间步 $t$：

- 第 1 层接收原始输入 $x_t$
- 第 2 层接收第 1 层的输出
- 第 3 层接收第 2 层的输出

所以更准确地说：

上一层的输出序列，会作为下一层的输入序列。

##### 7.3 output 的 shape 会变吗

输出仍然是：

```python
output.shape = (2, 3, 5)
```

也就是：

$(B,\ T,\ d_h)$

这里最容易误解的点：

PyTorch 的 `output` 只返回：

最后一层在所有时间步的输出。

不是把每一层都拼接起来返回。

##### 7.4 h_n 的 shape 会变

此时：

```python
h_n.shape = (3, 2, 5)
```

也就是：

$(num\_layers,\ B,\ d_h)$

因为它保存了每一层最后一个时间步的隐藏状态：

- Layer 1 的最后状态
- Layer 2 的最后状态
- Layer 3 的最后状态

#### 8. 双向 GRU 的维度变化

##### 8.1 最基础定义

```python
gru = nn.GRU(
    input_size=4,
    hidden_size=5,
    bidirectional=True,
    batch_first=True
)
```

##### 8.2 双向 GRU 的核心理解

双向 GRU 本质上就是：

两个独立的 GRU，同时从左到右、从右到左处理序列。

然后把两个方向的输出拼接起来，形成更完整的上下文表示。

##### 8.3 output 的 shape

此时：

```python
output.shape = (2, 3, 10)
```

也就是：

$(B,\ T,\ 2d_h)$

为什么最后一维变成 2 倍？

因为双向 GRU 会在每个时间步产生两个隐藏状态：

- 正向隐藏状态
- 反向隐藏状态

然后把它们拼接起来。

所以输出维度翻倍。

##### 8.4 h_n 的 shape

此时：

```python
h_n.shape = (2, 2, 5)
```

也就是：

$(num\_directions,\ B,\ d_h)$

这里第一维的 `2` 表示两个方向：

- forward
- backward

#### 9. 多层 + 双向 GRU 的维度变化

##### 9.1 设定

```python
gru = nn.GRU(
    input_size=4,
    hidden_size=5,
    num_layers=3,
    bidirectional=True,
    batch_first=True
)
```

##### 9.2 output 的 shape

此时：

```python
output.shape = (2, 3, 10)
```

也就是：

$(B,\ T,\ 2d_h)$

注意：

`output` 的最后一维只和方向数有关，不和层数有关。

因为 `output` 仍然只返回最后一层的所有时间步输出，但这一层是双向的，所以最后一维是 $2d_h$。

##### 9.3 h_n 的 shape

此时：

```python
h_n.shape = (6, 2, 5)
```

因为：

$6 = 3 \times 2$

所以一般规律是：

```python
h_n.shape = (num_layers * num_directions, B, d_h)
```

##### 9.4 h_n 的真实排列结构

它的第一维可以理解成：

- layer1_forward
- layer1_backward
- layer2_forward
- layer2_backward
- layer3_forward
- layer3_backward

所以第一维不是“纯层数”，而是：

层数 × 方向数 的展开结果。

#### 10. GRU 和 LSTM 在 PyTorch 实现上的对比

##### 10.1 相同点

GRU 和 LSTM 在 PyTorch 中有很多共同点：

- 都把多个门的线性变换合并起来
- 都使用 `weight_ih` 和 `weight_hh`
- 都支持多层和双向

##### 10.2 不同点

LSTM：

- 有 4 组门相关计算
- 所以权重第一维是 $4 \times hidden\_size$
- 返回 `output, (h_n, c_n)`

GRU：

- 有 3 组门相关计算
- 所以权重第一维是 $3 \times hidden\_size$
- 返回 `output, h_n`

##### 10.3 最本质的区别

LSTM 有两个状态：

- 隐藏状态
- 细胞状态

GRU 只有一个隐藏状态。

所以在 PyTorch 的返回值上：

- LSTM 会多一个 `c_n`
- GRU 没有 `c_n`

这和它们的理论结构完全对应。